# Audio Enhancement Pipeline Comparison

Interactive analysis of audio enhancement pipelines for the Reading Room BKK retrospective project.

**Source**: `pilot/quality_report.json` — DNSMOS scores across 3 pilot files and 8 pipelines.

In [ ]:
import json
import sys
from pathlib import Path

import altair as alt
import polars as pl

# Project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

REPORT_PATH = ROOT / "pilot" / "quality_report.json"
print(f"Project root: {ROOT}")
print(f"Report exists: {REPORT_PATH.exists()}")

## 1. Load Pilot Results

In [ ]:
with open(REPORT_PATH) as f:
    report = json.load(f)

# Transform nested JSON into tidy long-format DataFrame
rows = []
for filename, pipelines in report.items():
    for pipeline, scores in pipelines.items():
        for metric, value in scores.items():
            if isinstance(value, (int, float)):
                rows.append({
                    "file": filename,
                    "pipeline": pipeline,
                    "metric": metric,
                    "value": float(value),
                })

df = pl.DataFrame(rows)
print(f"Records: {len(df)}")
print(f"Files: {df['file'].unique().to_list()}")
print(f"Pipelines: {df['pipeline'].unique().to_list()}")
print(f"Metrics: {df['metric'].unique().to_list()}")
df.head(10)

## 2. DNSMOS Overview — Grouped Bar Charts

DNSMOS scores (1–5 scale): higher is better.
- **SIG**: Speech signal quality
- **BAK**: Background noise quality
- **OVRL**: Overall quality

In [ ]:
# Filter to DNSMOS metrics only
dnsmos_metrics = ["dnsmos_sig", "dnsmos_bak", "dnsmos_ovrl"]
metric_labels = {
    "dnsmos_sig": "Signal (SIG)",
    "dnsmos_bak": "Background (BAK)",
    "dnsmos_ovrl": "Overall (OVRL)",
}

df_dnsmos = (
    df.filter(pl.col("metric").is_in(dnsmos_metrics))
    .with_columns(
        pl.col("metric").replace(metric_labels).alias("metric_label")
    )
)

# Pipeline ordering (original first, then by average OVRL descending)
pipeline_order = [
    "original", "deepfilter_full", "hybrid_demucs_df", "frcrn_16k",
    "mossformer2_48k", "deepfilter_12dB", "demucs_vocals", "ffmpeg_gentle",
]

chart = (
    alt.Chart(df_dnsmos.to_pandas())
    .mark_bar()
    .encode(
        x=alt.X("pipeline:N", sort=pipeline_order, title=None,
                axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("value:Q", title="DNSMOS Score", scale=alt.Scale(domain=[0, 4])),
        color=alt.Color("pipeline:N", sort=pipeline_order,
                        legend=alt.Legend(title="Pipeline")),
        column=alt.Column("metric_label:N", title=None,
                          sort=["Signal (SIG)", "Background (BAK)", "Overall (OVRL)"]),
        row=alt.Row("file:N", title="File"),
        tooltip=["pipeline", "metric_label", alt.Tooltip("value:Q", format=".3f")],
    )
    .properties(width=220, height=140, title="DNSMOS Scores by Pipeline and File")
)

chart

## 3. Overall Score Comparison — Average Across Files

In [ ]:
# Average DNSMOS OVRL across all 3 pilot files
df_avg = (
    df.filter(pl.col("metric") == "dnsmos_ovrl")
    .group_by("pipeline")
    .agg(pl.col("value").mean().alias("avg_ovrl"))
    .sort("avg_ovrl", descending=True)
)

chart_avg = (
    alt.Chart(df_avg.to_pandas())
    .mark_bar()
    .encode(
        x=alt.X("avg_ovrl:Q", title="Average DNSMOS Overall Score",
                scale=alt.Scale(domain=[0, 3])),
        y=alt.Y("pipeline:N", sort="-x", title=None),
        color=alt.condition(
            alt.datum.pipeline == "original",
            alt.value("#999999"),
            alt.Color("pipeline:N", legend=None),
        ),
        tooltip=["pipeline", alt.Tooltip("avg_ovrl:Q", format=".3f")],
    )
    .properties(width=500, height=250,
                title="Average DNSMOS Overall Score (higher = better)")
)

# Add text labels
text = chart_avg.mark_text(align="left", dx=5).encode(
    text=alt.Text("avg_ovrl:Q", format=".3f")
)

chart_avg + text

## 4. Radar Chart — Multi-Dimensional Quality Profile

Compare how each pipeline performs across all DNSMOS dimensions (averaged across files).

In [ ]:
import math

# Average scores per pipeline per metric
df_radar_src = (
    df.filter(pl.col("metric").is_in(["dnsmos_p808", "dnsmos_sig", "dnsmos_bak", "dnsmos_ovrl"]))
    .group_by(["pipeline", "metric"])
    .agg(pl.col("value").mean().alias("avg_value"))
)

radar_labels = {
    "dnsmos_p808": "P.808",
    "dnsmos_sig": "Signal",
    "dnsmos_bak": "Background",
    "dnsmos_ovrl": "Overall",
}
metrics_ordered = list(radar_labels.keys())
n_metrics = len(metrics_ordered)

# Build radar polygon data
radar_rows = []
for pipeline in df_radar_src["pipeline"].unique().to_list():
    pipe_data = df_radar_src.filter(pl.col("pipeline") == pipeline)
    values = {}
    for row in pipe_data.iter_rows(named=True):
        values[row["metric"]] = row["avg_value"]

    for i, metric in enumerate(metrics_ordered):
        angle = 2 * math.pi * i / n_metrics - math.pi / 2
        val = values.get(metric, 0) / 4.0  # Normalize to 0-1 range (max score ~4)
        radar_rows.append({
            "pipeline": pipeline,
            "metric": radar_labels[metric],
            "x": val * math.cos(angle),
            "y": val * math.sin(angle),
            "value": values.get(metric, 0),
            "order": i,
        })
    # Close the polygon
    first_metric = metrics_ordered[0]
    angle = -math.pi / 2
    val = values.get(first_metric, 0) / 4.0
    radar_rows.append({
        "pipeline": pipeline,
        "metric": radar_labels[first_metric],
        "x": val * math.cos(angle),
        "y": val * math.sin(angle),
        "value": values.get(first_metric, 0),
        "order": n_metrics,
    })

df_radar = pl.DataFrame(radar_rows)

# Selected pipelines for clarity (top performers + baseline)
selected = ["original", "deepfilter_full", "hybrid_demucs_df", "mossformer2_48k", "frcrn_16k"]

radar_chart = (
    alt.Chart(df_radar.filter(pl.col("pipeline").is_in(selected)).to_pandas())
    .mark_line(point=True, opacity=0.7)
    .encode(
        x=alt.X("x:Q", axis=None),
        y=alt.Y("y:Q", axis=None),
        color=alt.Color("pipeline:N", legend=alt.Legend(title="Pipeline")),
        order="order:Q",
        tooltip=["pipeline", "metric", alt.Tooltip("value:Q", format=".3f")],
    )
    .properties(width=400, height=400, title="DNSMOS Quality Profile (Radar)")
)

# Add axis labels
label_rows = []
for i, metric in enumerate(metrics_ordered):
    angle = 2 * math.pi * i / n_metrics - math.pi / 2
    label_rows.append({
        "label": radar_labels[metric],
        "x": 1.15 * math.cos(angle),
        "y": 1.15 * math.sin(angle),
    })

labels = (
    alt.Chart(pl.DataFrame(label_rows).to_pandas())
    .mark_text(fontSize=12, fontWeight="bold")
    .encode(x="x:Q", y="y:Q", text="label:N")
)

radar_chart + labels

## 5. Per-File Heatmap

Detailed score matrix: pipeline (rows) vs metric (columns) for each file.

In [ ]:
df_heat = (
    df.filter(pl.col("metric").is_in(dnsmos_metrics))
    .with_columns(
        pl.col("metric").replace(metric_labels).alias("metric_label")
    )
)

heatmap = (
    alt.Chart(df_heat.to_pandas())
    .mark_rect()
    .encode(
        x=alt.X("metric_label:N", title=None,
                sort=["Signal (SIG)", "Background (BAK)", "Overall (OVRL)"]),
        y=alt.Y("pipeline:N", sort=pipeline_order, title=None),
        color=alt.Color("value:Q", scale=alt.Scale(scheme="viridis", domain=[0.8, 3.6]),
                        title="Score"),
        facet=alt.Facet("file:N", columns=3, title=None),
        tooltip=["pipeline", "metric_label", alt.Tooltip("value:Q", format=".3f")],
    )
    .properties(width=180, height=200, title="")
)

# Add score text
text = (
    alt.Chart(df_heat.to_pandas())
    .mark_text(fontSize=10)
    .encode(
        x=alt.X("metric_label:N",
                sort=["Signal (SIG)", "Background (BAK)", "Overall (OVRL)"]),
        y=alt.Y("pipeline:N", sort=pipeline_order),
        text=alt.Text("value:Q", format=".2f"),
        color=alt.condition(
            alt.datum.value > 2.0,
            alt.value("black"),
            alt.value("white"),
        ),
        facet=alt.Facet("file:N", columns=3, title=None),
    )
    .properties(width=180, height=200)
)

(heatmap + text).resolve_scale(color="shared")

## 6. Improvement Over Baseline

How much does each pipeline improve over the original (unprocessed) audio?

In [ ]:
# Calculate improvement (delta) over original for each file/pipeline
df_ovrl = df.filter(pl.col("metric") == "dnsmos_ovrl")

# Get original scores per file
originals = (
    df_ovrl.filter(pl.col("pipeline") == "original")
    .select(["file", pl.col("value").alias("original_value")])
)

df_delta = (
    df_ovrl.filter(pl.col("pipeline") != "original")
    .join(originals, on="file")
    .with_columns(
        (pl.col("value") - pl.col("original_value")).alias("improvement")
    )
)

improvement_chart = (
    alt.Chart(df_delta.to_pandas())
    .mark_bar()
    .encode(
        x=alt.X("pipeline:N", sort=pipeline_order[1:], title=None,
                axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("improvement:Q", title="DNSMOS OVRL Improvement"),
        color=alt.Color("file:N", title="File"),
        xOffset="file:N",
        tooltip=["pipeline", "file",
                 alt.Tooltip("improvement:Q", format="+.3f"),
                 alt.Tooltip("value:Q", format=".3f", title="Final"),
                 alt.Tooltip("original_value:Q", format=".3f", title="Original")],
    )
    .properties(width=600, height=300,
                title="DNSMOS Overall Improvement Over Baseline")
)

improvement_chart

## 7. Summary Table

In [ ]:
# Pivot to wide format for readable summary
summary = (
    df.filter(pl.col("metric").is_in(["dnsmos_sig", "dnsmos_bak", "dnsmos_ovrl"]))
    .group_by(["pipeline", "metric"])
    .agg(pl.col("value").mean().round(3).alias("avg"))
    .pivot(on="metric", index="pipeline", values="avg")
    .sort("dnsmos_ovrl", descending=True)
    .rename({
        "dnsmos_sig": "Avg SIG",
        "dnsmos_bak": "Avg BAK",
        "dnsmos_ovrl": "Avg OVRL",
    })
)

summary

## 8. Audio Playback (A/B Listening)

Listen to original vs enhanced audio for direct comparison.
Requires pilot audio files in `pilot/original/` and `pilot/models/`.

In [ ]:
from IPython.display import Audio, display, HTML

PILOT_DIR = ROOT / "pilot"
PILOT_FILES = ["01_earliest_talk", "02_screening_talk", "03_thai_talk"]
COMPARE_PIPELINES = ["original", "hybrid_demucs_df", "deepfilter_full", "mossformer2_48k"]

for fname in PILOT_FILES:
    display(HTML(f"<h3>{fname}</h3>"))
    for pipe in COMPARE_PIPELINES:
        if pipe == "original":
            path = PILOT_DIR / "original" / f"{fname}.wav"
        else:
            path = PILOT_DIR / "models" / pipe / f"{fname}.wav"

        if path.exists():
            display(HTML(f"<b>{pipe}</b>"))
            display(Audio(filename=str(path)))
        else:
            display(HTML(f"<b>{pipe}</b> — file not found: {path}"))

## 9. Run New Enhancement (Interactive)

Enhance a new file and score it immediately.

In [ ]:
# Uncomment and modify to run enhancement on a new file:

# from scripts.audio.enhance import PIPELINES
# from scripts.audio.score import score_audio
#
# input_wav = "path/to/your/audio.wav"
# output_wav = "path/to/output.wav"
#
# # Enhance with preferred pipeline
# PIPELINES["hybrid_demucs_df"](input_wav, output_wav)
#
# # Score the result
# scores = score_audio(output_wav)
# print(f"DNSMOS Overall: {scores.get('dnsmos_ovrl', 'N/A'):.3f}")
# print(f"DNSMOS Signal:  {scores.get('dnsmos_sig', 'N/A'):.3f}")
# print(f"DNSMOS Background: {scores.get('dnsmos_bak', 'N/A'):.3f}")